# Python  Industrial / Production Patterns

---

This notebook covers Python patterns and tools used in **real-world, production AI/ML codebases**. These are the patterns you see in PyTorch, Hugging Face Transformers, FastAPI, and large-scale ML pipelines.

## Table of Contents
1. [Type Hints & Pydantic](#1)
2. [Decorators  Advanced](#2)
3. [Context Managers](#3)
4. [Logging](#4)
5. [Testing with pytest](#5)
6. [Design Patterns in ML](#6)
7. [Concurrency  Threading & Multiprocessing](#7)
8. [Async / Await](#8)
9. [Dataclasses](#9)
10. [Packaging & Project Structure](#10)
11. [Environment & Config Management](#11)
12. [Profiling & Performance](#12)
13. [Additional Learning Resources](#13)

---
## 1. Type Hints & Pydantic <a id='1'></a>

Type hints make code **self-documenting** and enable static analysis tools (mypy, pyright). Pydantic adds **runtime validation**  essential for API request/response models and ML configs.

Type hints do NOT enforce types at runtime by default  they are hints for tools and humans.

In [1]:
from typing import List, Dict, Optional, Tuple, Union, Any, Callable, Iterator, Generator
from typing import TypeVar, Generic

# ===== Basic type hints =====
def train(
    model_name: str,
    data: List[List[float]],
    labels: List[int],
    lr: float = 1e-3,
    epochs: int = 10,
    callbacks: Optional[List[Callable]] = None
) -> Dict[str, float]:
    """Train a model and return metrics."""
    return {'accuracy': 0.95, 'loss': 0.12}

# ===== Union  multiple allowed types =====
def load_model(path: Union[str, 'Path']) -> Any:
    pass

# Python 3.10+  use | instead of Union
def predict(x: float | int) -> float:
    return float(x)

# ===== TypeVar  generic functions =====
T = TypeVar('T')

def first(items: List[T]) -> T:
    return items[0]

print(first([1, 2, 3]))      # int
print(first(['a', 'b']))     # str

# ===== Pydantic  data validation =====
try:
    from pydantic import BaseModel, Field, validator

    class TrainingConfig(BaseModel):
        model_name: str
        learning_rate: float = Field(default=1e-3, gt=0, lt=1)
        batch_size: int = Field(default=32, ge=1)
        epochs: int = Field(default=10, ge=1)
        dropout: float = Field(default=0.1, ge=0, le=1)
        hidden_dims: List[int] = [256, 128]
        optimizer: str = 'adam'

        @validator('optimizer')
        def validate_optimizer(cls, v):
            allowed = ['adam', 'sgd', 'rmsprop', 'adagrad']
            if v not in allowed:
                raise ValueError(f'optimizer must be one of {allowed}')
            return v

        class Config:
            env_prefix = 'MODEL_'  # reads MODEL_LEARNING_RATE from env

    cfg = TrainingConfig(model_name='BERT', learning_rate=0.001, batch_size=64)
    print(cfg.dict())
    print(cfg.json(indent=2))

    # Pydantic catches invalid data at creation time
    try:
        bad = TrainingConfig(model_name='X', learning_rate=5.0)  # lr > 1
    except Exception as e:
        print(f'Validation error: {e}')

except ImportError:
    print('Install pydantic: pip install pydantic')

1
a
Install pydantic: pip install pydantic


---
## 2. Decorators  Advanced <a id='2'></a>

Decorators are **higher-order functions** that wrap another function to add behavior. They are used extensively in frameworks: `@app.route` (Flask/FastAPI), `@torch.no_grad()` (PyTorch), `@property` (Python), `@staticmethod`.

In [2]:
import time
import functools
import logging

# ===== Decorator that preserves function metadata =====
def retry(max_attempts: int = 3, delay: float = 1.0):
    """Decorator factory  decorator with arguments."""
    def decorator(func):
        @functools.wraps(func)  # preserves __name__, __doc__, etc.
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f'Attempt {attempt} failed: {e}. Retrying in {delay}s...')
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.1)
def call_api(endpoint: str):
    import random
    if random.random() < 0.7:  # 70% failure rate for demo
        raise ConnectionError('API unavailable')
    return {'data': 'response'}

try:
    result = call_api('/predict')
    print(result)
except ConnectionError:
    print('All retries exhausted')

# ===== Cache decorator =====
def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache  # expose cache
    return wrapper

@memoize
def expensive_embedding(word: str):
    time.sleep(0.001)  # simulate expensive computation
    return hash(word) % 1000  # dummy embedding

# ===== Class-based decorator =====
class RateLimit:
    def __init__(self, calls_per_second: float):
        self.min_interval = 1.0 / calls_per_second
        self.last_call = 0.0

    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            elapsed = time.time() - self.last_call
            if elapsed < self.min_interval:
                time.sleep(self.min_interval - elapsed)
            self.last_call = time.time()
            return func(*args, **kwargs)
        return wrapper

@RateLimit(calls_per_second=2)
def fetch_data():
    return 'data'

# ===== Stacking decorators =====
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f'Calling {func.__name__}')
        result = func(*args, **kwargs)
        print(f'{func.__name__} returned {result}')
        return result
    return wrapper

def validate_input(func):
    @functools.wraps(func)
    def wrapper(x, *args, **kwargs):
        if x < 0:
            raise ValueError('Input must be non-negative')
        return func(x, *args, **kwargs)
    return wrapper

@log_call
@validate_input  # applied first (innermost)
def sqrt_func(x: float) -> float:
    return x ** 0.5

print(sqrt_func(16))

Attempt 1 failed: API unavailable. Retrying in 0.1s...
Attempt 2 failed: API unavailable. Retrying in 0.1s...
All retries exhausted
Calling sqrt_func
sqrt_func returned 4.0
4.0


---
## 3. Context Managers <a id='3'></a>

Context managers (`with` statement) guarantee **setup and teardown**  file handles, database connections, GPU memory, timing blocks.

In [3]:
import time
from contextlib import contextmanager, suppress

# ===== Class-based context manager =====
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f'Elapsed: {self.elapsed:.4f}s')
        return False  # don't suppress exceptions

with Timer() as t:
    total = sum(range(1_000_000))
print(f'Sum: {total}, Time: {t.elapsed:.4f}s')

# ===== Generator-based context manager =====
@contextmanager
def managed_experiment(name: str):
    print(f'Starting experiment: {name}')
    metrics = {}
    try:
        yield metrics  # everything before yield is __enter__
    except Exception as e:
        print(f'Experiment failed: {e}')
        raise
    finally:
        print(f'Experiment {name} finished. Metrics: {metrics}')

with managed_experiment('baseline') as m:
    m['accuracy'] = 0.94
    m['loss'] = 0.15

# ===== suppress  ignore specific exceptions =====
with suppress(FileNotFoundError):
    import os
    os.remove('nonexistent_file.txt')  # no error raised

# ===== Nested context managers =====
@contextmanager
def gpu_memory_manager():
    print('Allocating GPU memory')
    yield
    print('Freeing GPU memory')

@contextmanager
def gradient_tape():
    print('Starting gradient recording')
    yield
    print('Stopping gradient recording')

with gpu_memory_manager(), gradient_tape():
    print('Running forward pass...')

Elapsed: 0.0225s
Sum: 499999500000, Time: 0.0225s
Starting experiment: baseline
Experiment baseline finished. Metrics: {'accuracy': 0.94, 'loss': 0.15}
Allocating GPU memory
Starting gradient recording
Running forward pass...
Stopping gradient recording
Freeing GPU memory


---
## 4. Logging <a id='4'></a>

**Never use `print` in production code.** Use the `logging` module  it supports log levels, file output, formatting, and is controllable without changing code.

In [4]:
import logging
import sys

# ===== Basic logging setup =====
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout),
        # logging.FileHandler('training.log')  # also log to file
    ]
)

logger = logging.getLogger(__name__)

# Log levels: DEBUG < INFO < WARNING < ERROR < CRITICAL
logger.debug('Batch loaded: shape (32, 512)')
logger.info('Epoch 1/50  loss: 0.452, acc: 0.841')
logger.warning('Learning rate very high: 0.9')
logger.error('NaN detected in gradients')
logger.critical('GPU out of memory  training halted')

# ===== Named loggers (per module) =====
train_logger = logging.getLogger('training')
data_logger  = logging.getLogger('data')

# ===== Production logging setup =====
def setup_logger(name: str, level=logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)

    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter(
            '%(asctime)s %(name)s %(levelname)s %(message)s'
        ))
        logger.addHandler(handler)
    return logger

log = setup_logger('ml_pipeline')
log.info('Pipeline started')

# ===== Logging exceptions with traceback =====
try:
    result = 1 / 0
except ZeroDivisionError:
    logger.exception('Division failed')  # logs traceback automatically

2026-06-19 13:07:33 | DEBUG    | __main__ | Batch loaded: shape (32, 512)
2026-06-19 13:07:33 | INFO     | __main__ | Epoch 1/50  loss: 0.452, acc: 0.841
2026-06-19 13:07:33 | WARNING  | __main__ | Learning rate very high: 0.9
2026-06-19 13:07:33 | ERROR    | __main__ | NaN detected in gradients
2026-06-19 13:07:33 | CRITICAL | __main__ | GPU out of memory  training halted


2026-06-19 13:07:33,663 ml_pipeline INFO Pipeline started


2026-06-19 13:07:33 | INFO     | ml_pipeline | Pipeline started
2026-06-19 13:07:33 | ERROR    | __main__ | Division failed
Traceback (most recent call last):
  File "/tmp/ipykernel_22971/4032682609.py", line 46, in <module>
    result = 1 / 0
             ~~^~~
ZeroDivisionError: division by zero


---
## 5. Testing with pytest <a id='5'></a>

Tests are not optional in production ML systems  they catch regressions, validate data pipelines, and ensure model behavior doesn't change unexpectedly.

In [5]:
# In a real project, save these in test_*.py files and run: pytest tests/

# ===== Basic test structure =====
def add(a, b):
    return a + b

def test_add():
    assert add(2, 3) == 5
    assert add(-1, 1) == 0
    assert add(0, 0) == 0

# ===== Testing with pytest fixtures (conceptual) =====
# In conftest.py:
# @pytest.fixture
# def sample_dataset():
#     X = [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]
#     y = [0, 1, 0]
#     return X, y
#
# def test_model_trains(sample_dataset):
#     X, y = sample_dataset
#     model = LinearRegression()
#     model.fit(X, y)
#     assert hasattr(model, 'coef_')

# ===== Parametrized tests =====
# @pytest.mark.parametrize('input,expected', [
#     (0,  0.5),
#     (1,  0.731),
#     (-1, 0.269),
# ])
# def test_sigmoid(input, expected):
#     import math
#     result = 1 / (1 + math.exp(-input))
#     assert abs(result - expected) < 0.001

# ===== Exception testing =====
# def test_invalid_input():
#     with pytest.raises(ValueError, match='must be positive'):
#         validate_batch_size(-1)

# ===== ML-specific test patterns =====
import math

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def test_sigmoid_bounds():
    assert 0 < sigmoid(0) < 1
    assert sigmoid(100) > 0.99
    assert sigmoid(-100) < 0.01

def test_sigmoid_symmetry():
    for x in [0.5, 1.0, 2.0, 5.0]:
        assert abs(sigmoid(x) + sigmoid(-x) - 1.0) < 1e-10

def test_normalization():
    data = [1.0, 2.0, 3.0, 4.0, 5.0]
    mean = sum(data) / len(data)
    std  = (sum((x - mean)**2 for x in data) / len(data)) ** 0.5
    normalized = [(x - mean) / std for x in data]
    assert abs(sum(normalized) / len(normalized)) < 1e-10  # mean ~ 0
    assert abs((sum(x**2 for x in normalized) / len(normalized)) ** 0.5 - 1.0) < 1e-10  # std ~ 1

# Run tests inline
test_add()
test_sigmoid_bounds()
test_sigmoid_symmetry()
test_normalization()
print('All tests passed!')

All tests passed!


---
## 6. Design Patterns in ML <a id='6'></a>

Design patterns are reusable solutions to common software problems. These specific ones appear constantly in ML frameworks.

| Pattern | Where you see it in ML |
|---------|------------------------|
| **Strategy** | Swap optimizers, loss functions, callbacks |
| **Factory** | Create models by name from config |
| **Observer** | Callbacks (on_epoch_end, on_batch_end) |
| **Pipeline** | Sklearn pipelines, data transforms |
| **Registry** | Plugin systems for model architectures |

In [6]:
from abc import ABC, abstractmethod
from typing import Dict, Type

# ===== Strategy Pattern  swap algorithms at runtime =====
class Optimizer(ABC):
    @abstractmethod
    def update(self, params, grads, lr): pass

class SGD(Optimizer):
    def update(self, params, grads, lr):
        return [p - lr * g for p, g in zip(params, grads)]

class AdamOptimizer(Optimizer):
    def __init__(self, beta1=0.9, beta2=0.999, eps=1e-8):
        self.beta1, self.beta2, self.eps = beta1, beta2, eps
        self.m, self.v, self.t = [], [], 0

    def update(self, params, grads, lr):
        self.t += 1
        if not self.m:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        updated = []
        for i, (p, g) in enumerate(zip(params, grads)):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g**2
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            updated.append(p - lr * m_hat / (v_hat**0.5 + self.eps))
        return updated

# ===== Factory Pattern  create objects by name =====
_OPTIMIZERS: Dict[str, Type[Optimizer]] = {
    'sgd':  SGD,
    'adam': AdamOptimizer,
}

def create_optimizer(name: str, **kwargs) -> Optimizer:
    if name not in _OPTIMIZERS:
        raise ValueError(f'Unknown optimizer: {name}. Choose from {list(_OPTIMIZERS)}')
    return _OPTIMIZERS[name](**kwargs)

opt = create_optimizer('adam', beta1=0.9)
print(type(opt).__name__)

# ===== Registry Pattern  auto-register subclasses =====
MODEL_REGISTRY: Dict[str, type] = {}

def register(name: str):
    def decorator(cls):
        MODEL_REGISTRY[name] = cls
        return cls
    return decorator

@register('resnet')
class ResNet:
    def forward(self, x): return f'ResNet output'

@register('bert')
class BERT:
    def forward(self, x): return f'BERT output'

def build_model(name: str):
    return MODEL_REGISTRY[name]()

model = build_model('bert')
print(model.forward(None))
print('Registered models:', list(MODEL_REGISTRY.keys()))

# ===== Observer Pattern  callbacks =====
class TrainingEvent:
    def __init__(self):
        self._callbacks = []

    def subscribe(self, callback):
        self._callbacks.append(callback)

    def emit(self, **data):
        for cb in self._callbacks:
            cb(**data)

on_epoch_end = TrainingEvent()
on_epoch_end.subscribe(lambda epoch, loss: print(f'Epoch {epoch}: loss={loss:.4f}'))
on_epoch_end.subscribe(lambda epoch, loss: print(f'Saving checkpoint at epoch {epoch}') if epoch % 5 == 0 else None)

for epoch in range(1, 8):
    on_epoch_end.emit(epoch=epoch, loss=1.0 / epoch)

AdamOptimizer
BERT output
Registered models: ['resnet', 'bert']
Epoch 1: loss=1.0000
Epoch 2: loss=0.5000
Epoch 3: loss=0.3333
Epoch 4: loss=0.2500
Epoch 5: loss=0.2000
Saving checkpoint at epoch 5
Epoch 6: loss=0.1667
Epoch 7: loss=0.1429


---
## 7. Concurrency  Threading & Multiprocessing <a id='7'></a>

| Approach | Best for | GIL affected? |
|----------|----------|---------------|
| **Threading** | I/O-bound tasks (API calls, file reads) | Yes (GIL limits CPU parallelism) |
| **Multiprocessing** | CPU-bound tasks (data preprocessing) | No (each process has own GIL) |
| **async/await** | Many concurrent I/O tasks | N/A (single thread) |

**GIL (Global Interpreter Lock)**: Python's mutex that prevents multiple threads from executing Python bytecode simultaneously. NumPy/PyTorch operations release the GIL, enabling real parallelism.

In [7]:
import concurrent.futures
import threading
import multiprocessing
import time

# ===== ThreadPoolExecutor  I/O bound tasks =====
def download_data(url: str) -> str:
    time.sleep(0.1)  # simulate network latency
    return f'Data from {url}'

urls = [f'https://api.example.com/data/{i}' for i in range(10)]

start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(download_data, url) for url in urls]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]
print(f'Downloaded {len(results)} items in {time.time()-start:.2f}s')

# ===== ProcessPoolExecutor  CPU bound tasks =====
def preprocess_batch(batch):
    # CPU-intensive preprocessing
    return [x**2 + x**0.5 for x in batch]

large_dataset = list(range(10000))
batches = [large_dataset[i:i+1000] for i in range(0, 10000, 1000)]

# Sequential
start = time.time()
seq_results = [preprocess_batch(b) for b in batches]
print(f'Sequential: {time.time()-start:.4f}s')

# Parallel
start = time.time()
with concurrent.futures.ProcessPoolExecutor() as executor:
    par_results = list(executor.map(preprocess_batch, batches))
print(f'Parallel:   {time.time()-start:.4f}s')

# ===== Thread-safe counter with Lock =====
class AtomicCounter:
    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()

    def increment(self):
        with self._lock:
            self._value += 1

    @property
    def value(self):
        return self._value

counter = AtomicCounter()
threads = [threading.Thread(target=counter.increment) for _ in range(100)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Counter: {counter.value}')  # 100

Downloaded 10 items in 0.20s
Sequential: 0.0020s
Parallel:   0.0462s
Counter: 100


---
## 8. Async / Await <a id='8'></a>

`asyncio` enables **cooperative multitasking**  while waiting for I/O, other tasks run. This is the foundation of FastAPI and modern LLM API clients.

In [8]:
import asyncio
import time

# ===== Basic async functions =====
async def fetch_embedding(text: str) -> list:
    await asyncio.sleep(0.1)  # simulate API call
    return [hash(c) % 100 for c in text[:5]]

async def main():
    # Sequential  0.3s total
    start = time.time()
    e1 = await fetch_embedding('hello')
    e2 = await fetch_embedding('world')
    e3 = await fetch_embedding('python')
    print(f'Sequential: {time.time()-start:.2f}s')

    # Concurrent  ~0.1s total (all run simultaneously)
    start = time.time()
    e1, e2, e3 = await asyncio.gather(
        fetch_embedding('hello'),
        fetch_embedding('world'),
        fetch_embedding('python')
    )
    print(f'Concurrent: {time.time()-start:.2f}s')

# Run in Jupyter
await main()

# ===== Async generator  streaming LLM responses =====
async def stream_tokens(text: str):
    for token in text.split():
        await asyncio.sleep(0.05)  # simulate token generation
        yield token

async def print_stream():
    async for token in stream_tokens('The quick brown fox jumps over the lazy dog'):
        print(token, end=' ', flush=True)
    print()

await print_stream()

# ===== Async context manager =====
class AsyncDBConnection:
    async def __aenter__(self):
        print('Connecting to database...')
        await asyncio.sleep(0.01)
        return self

    async def __aexit__(self, *args):
        print('Closing connection...')

    async def query(self, sql: str):
        await asyncio.sleep(0.01)
        return f'Results for: {sql}'

async def use_db():
    async with AsyncDBConnection() as db:
        result = await db.query('SELECT * FROM models')
        print(result)

await use_db()

Sequential: 0.30s
Concurrent: 0.10s
The quick brown fox jumps over the lazy dog 
Connecting to database...
Results for: SELECT * FROM models
Closing connection...


---
## 9. Dataclasses <a id='9'></a>

Dataclasses auto-generate `__init__`, `__repr__`, `__eq__` from field definitions. They are lighter than Pydantic but have no runtime validation.

In [9]:
from dataclasses import dataclass, field, asdict, astuple
from typing import List, Optional

@dataclass
class ModelConfig:
    name: str
    hidden_dim: int = 256
    num_layers: int = 4
    dropout: float = 0.1
    layer_dims: List[int] = field(default_factory=lambda: [256, 128, 64])
    vocab_size: Optional[int] = None

    def total_params(self) -> int:
        return sum(self.layer_dims)

@dataclass(frozen=True)  # immutable  can be used as dict key
class DataPoint:
    features: tuple
    label: int

@dataclass(order=True)  # enables <, >, ==
class Metric:
    sort_index: float = field(init=False, repr=False)
    name: str
    value: float

    def __post_init__(self):
        self.sort_index = self.value  # sort by value

cfg = ModelConfig(name='Transformer')
print(cfg)                   # auto __repr__
print(asdict(cfg))           # convert to dict
print(cfg.total_params())

metrics = [
    Metric('accuracy', 0.94),
    Metric('f1', 0.91),
    Metric('recall', 0.88),
]
print(sorted(metrics, reverse=True))  # sort by value

ModelConfig(name='Transformer', hidden_dim=256, num_layers=4, dropout=0.1, layer_dims=[256, 128, 64], vocab_size=None)
{'name': 'Transformer', 'hidden_dim': 256, 'num_layers': 4, 'dropout': 0.1, 'layer_dims': [256, 128, 64], 'vocab_size': None}
448
[Metric(name='accuracy', value=0.94), Metric(name='f1', value=0.91), Metric(name='recall', value=0.88)]


---
## 10. Packaging & Project Structure <a id='10'></a>

A well-structured ML project looks like this:

```
my_ml_project/
├── src/
│   └── my_package/
│       ├── __init__.py
│       ├── models/
│       │   ├── __init__.py
│       │   ├── base.py
│       │   └── transformer.py
│       ├── data/
│       │   ├── __init__.py
│       │   ├── dataset.py
│       │   └── preprocessing.py
│       ├── training/
│       │   ├── __init__.py
│       │   ├── trainer.py
│       │   └── callbacks.py
│       └── utils/
│           ├── __init__.py
│           └── metrics.py
├── tests/
│   ├── conftest.py
│   ├── test_models.py
│   └── test_data.py
├── notebooks/
├── configs/
│   └── train_config.yaml
├── pyproject.toml      ← modern Python packaging
├── requirements.txt
├── setup.py            ← legacy (optional)
└── README.md
```

In [10]:
# pyproject.toml example (modern standard):
pyproject_toml = '''
[build-system]
requires = ["setuptools>=61", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "my-ml-project"
version = "0.1.0"
description = "AI/ML project"
requires-python = ">=3.9"
dependencies = [
    "numpy>=1.24",
    "torch>=2.0",
    "transformers>=4.30",
    "pydantic>=2.0",
]

[project.optional-dependencies]
dev = ["pytest", "black", "ruff", "mypy"]
'''
print(pyproject_toml)

# __init__.py  control what gets exported from your package
init_example = '''
# src/my_package/__init__.py
from .models.transformer import TransformerModel
from .data.dataset import MLDataset
from .training.trainer import Trainer

__version__ = "0.1.0"
__all__ = ["TransformerModel", "MLDataset", "Trainer"]
'''
print(init_example)


[build-system]
requires = ["setuptools>=61", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "my-ml-project"
version = "0.1.0"
description = "AI/ML project"
requires-python = ">=3.9"
dependencies = [
    "numpy>=1.24",
    "torch>=2.0",
    "transformers>=4.30",
    "pydantic>=2.0",
]

[project.optional-dependencies]
dev = ["pytest", "black", "ruff", "mypy"]


# src/my_package/__init__.py
from .models.transformer import TransformerModel
from .data.dataset import MLDataset
from .training.trainer import Trainer

__version__ = "0.1.0"
__all__ = ["TransformerModel", "MLDataset", "Trainer"]



---
## 11. Environment & Config Management <a id='11'></a>

In [11]:
import os
from pathlib import Path

# ===== Environment variables  never hardcode secrets =====
# Load from .env file using python-dotenv
try:
    from dotenv import load_dotenv
    load_dotenv()  # loads .env file into os.environ
except ImportError:
    pass

API_KEY   = os.environ.get('OPENAI_API_KEY', 'not-set')
DB_URL    = os.environ.get('DATABASE_URL', 'sqlite:///local.db')
DEBUG     = os.environ.get('DEBUG', 'false').lower() == 'true'
MAX_BATCH = int(os.environ.get('MAX_BATCH_SIZE', '32'))

print(f'Debug mode: {DEBUG}')
print(f'Max batch: {MAX_BATCH}')

# ===== YAML config files =====
try:
    import yaml

    config_yaml = '''
    model:
      name: bert-base
      hidden_dim: 768
      num_layers: 12

    training:
      learning_rate: 2.0e-5
      batch_size: 32
      epochs: 3
      warmup_steps: 500

    data:
      train_path: data/train.csv
      val_path: data/val.csv
    '''
    config = yaml.safe_load(config_yaml)
    print(config['model']['name'])
    print(config['training']['learning_rate'])
except ImportError:
    print('Install pyyaml: pip install pyyaml')

Debug mode: False
Max batch: 32
bert-base
2e-05


---
## 12. Profiling & Performance <a id='12'></a>

In [12]:
import timeit
import cProfile
import pstats
import io

# ===== timeit  benchmark small code snippets =====
# List comprehension vs loop
list_comp = timeit.timeit('[x**2 for x in range(1000)]', number=10000)
loop      = timeit.timeit('''
result = []
for x in range(1000):
    result.append(x**2)
''', number=10000)

print(f'List comp: {list_comp:.3f}s')
print(f'Loop:      {loop:.3f}s')
print(f'Speedup:   {loop/list_comp:.2f}x')

# ===== cProfile  find bottlenecks =====
def slow_function():
    total = 0
    for i in range(10000):
        for j in range(100):
            total += i * j
    return total

pr = cProfile.Profile()
pr.enable()
slow_function()
pr.disable()

sio = io.StringIO()
ps  = pstats.Stats(pr, stream=sio).sort_stats('cumulative')
ps.print_stats(5)  # top 5 slowest functions
print(sio.getvalue())

# ===== Memory efficiency tips =====
import sys

lst = list(range(10000))
gen = (x for x in range(10000))

print(f'List size:      {sys.getsizeof(lst):,} bytes')
print(f'Generator size: {sys.getsizeof(gen):,} bytes')

# Use __slots__ to reduce memory per instance
class WithSlots:
    __slots__ = ['x', 'y', 'z']
    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z

class WithoutSlots:
    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z

a = WithSlots(1, 2, 3)
b = WithoutSlots(1, 2, 3)
print(f'With slots:    {sys.getsizeof(a)} bytes')
print(f'Without slots: {sys.getsizeof(b)} bytes')

List comp: 0.866s
Loop:      0.863s
Speedup:   1.00x
         180 function calls (177 primitive calls) in 0.078 seconds

   Ordered by: cumulative time
   List reduced from 81 to 5 due to restriction <5>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.005    0.002    0.078    0.039 /home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3712(run_code)
        2    0.000    0.000    0.072    0.036 {built-in method builtins.exec}
        1    0.072    0.072    0.072    0.072 /tmp/ipykernel_22971/257891885.py:20(slow_function)
        1    0.000    0.000    0.000    0.000 /usr/lib/python3.12/asyncio/events.py:86(_run)
        1    0.000    0.000    0.000    0.000 {method 'run' of '_contextvars.Context' objects}



List size:      80,056 bytes
Generator size: 192 bytes
With slots:    56 bytes
Without slots: 48 bytes


---
## 13. Additional Learning Resources <a id='13'></a>

### Design Patterns & Architecture
- **Clean Code in Python** (Mariano Anaya): https://www.packtpub.com/product/clean-code-in-python/9781788835831
- **Python Design Patterns** (Real Python): https://realpython.com/python-design-patterns/
- **Architecture Patterns with Python** (Percival & Gregory): https://www.cosmicpython.com/book/preface.html (FREE)

### Type Hints & Validation
- **mypy Documentation**: https://mypy.readthedocs.io/en/stable/
- **Pydantic Documentation**: https://docs.pydantic.dev/
- **PEP 484  Type Hints**: https://peps.python.org/pep-0484/

### Testing
- **pytest Documentation**: https://docs.pytest.org/en/stable/
- **Testing ML Systems** (Google): https://research.google/pubs/pub46555/
- **Great Expectations (data validation)**: https://greatexpectations.io/

### Concurrency & Async
- **asyncio Documentation**: https://docs.python.org/3/library/asyncio.html
- **Trio Tutorial**: https://trio.readthedocs.io/
- **Python Concurrency** (Real Python): https://realpython.com/python-concurrency/

### Performance
- **High Performance Python** (Gorelick & Ozsvald): https://www.oreilly.com/library/view/high-performance-python/9781492055013/
- **Python Profiling** (Real Python): https://realpython.com/python-profiling/

### Production Python
- **Effective Python** (Brett Slatkin): https://effectivepython.com/
- **Python Packaging Guide**: https://packaging.python.org/en/latest/
- **Hypermodern Python** (Claudio Jolowicz): https://cjolowicz.github.io/posts/hypermodern-python-01-setup/